In [1]:
from pyspark.sql import functions as F
import os
from pyspark.sql.functions import explode_outer, explode
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode_outer
import os
import sys


spark = SparkSession.builder \
    .appName("JupyterPySpark").getOrCreate()
# Total row count target
TOTAL_RECORDS = 1_000_000

# 1. Create a distributed DataFrame skeleton using spark.range
# This generates 1M rows across cluster partitions automatically
raw_id_df = spark.range(1, TOTAL_RECORDS + 1)

# 2. Enrich the skeleton with random, diverse dataset variables
mock_etl_df = raw_id_df.withColumn(
    "transaction_id", 
    F.concat(F.lit("TXN_"), F.col("id"))
).withColumn(
    "customer_id", 
    # Generates a random customer ID between 1000 and 5000; injects ~2% NULL values
    F.when(F.rand() < 0.02, F.lit(None)).otherwise(F.concat(F.lit("CUST_"), F.expr("cast(rand() * 4000 + 1000 as int)")))
).withColumn(
    "image_name", 
    # Cycles through 5 generated image IDs to align with our earlier mockup storage
    F.concat(F.lit("img_00"), F.expr("cast(rand() * 5 + 1 as int)"), F.lit(".jpg"))
).withColumn(
    "amount", 
    # Generates transaction values between -$10 and $500 to simulate bad negative price inputs (~1% negative)
    F.round(F.expr("rand() * 510 - 10"), 2)
).withColumn(
    "timestamp", 
    # Distributes transactions over a historical window of seconds
    F.to_timestamp(F.expr("cast(1786851200 + (rand() * 86400 * 30) as timestamp)"))
).drop("id") # Drop the increment index column

# 3. Cache the execution or write it down as a parquet target
TARGET_PATH = os.getcwd()

# Write data out split by partitions to maximize read speeds later
# mock_etl_df.write.mode("overwrite").parquet(TARGET_PATH)

print(f"✅ Successfully generated and saved {TOTAL_RECORDS:,} mock records to {TARGET_PATH}")


c:\Users\uday.bachina\airflow_folder\airflow_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


✅ Successfully generated and saved 1,000,000 mock records to c:\Users\uday.bachina\airflow_folder\dags


In [2]:
print(mock_etl_df.show())
print(mock_etl_df.count())

+--------------+-----------+-----------+------+--------------------+
|transaction_id|customer_id| image_name|amount|           timestamp|
+--------------+-----------+-----------+------+--------------------+
|         TXN_1|  CUST_3673|img_005.jpg|400.16|2026-09-15 07:45:...|
|         TXN_2|  CUST_1159|img_003.jpg|232.09|2026-08-19 09:23:...|
|         TXN_3|  CUST_4530|img_003.jpg| 59.02|2026-08-18 14:05:...|
|         TXN_4|  CUST_1516|img_001.jpg| 40.36|2026-08-28 10:35:...|
|         TXN_5|  CUST_4792|img_005.jpg|108.95|2026-09-11 20:19:...|
|         TXN_6|  CUST_4384|img_002.jpg| 19.19|2026-09-05 03:07:...|
|         TXN_7|  CUST_2528|img_004.jpg|222.73|2026-08-23 19:32:...|
|         TXN_8|  CUST_3514|img_002.jpg|179.79|2026-09-14 03:54:...|
|         TXN_9|  CUST_3386|img_003.jpg| -2.03|2026-09-12 20:56:...|
|        TXN_10|  CUST_3698|img_005.jpg| 74.96|2026-09-10 16:10:...|
|        TXN_11|  CUST_3476|img_001.jpg|210.16|2026-08-29 17:05:...|
|        TXN_12|  CUST_4656|img_00

In [3]:
mock_etl_df.toPandas().to_csv("data_set.csv",header=True,index=False)

c:\Users\uday.bachina\airflow_folder\airflow_env\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\uday.bachina\airflow_folder\airflow_env\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
df= spark.read.csv('data_set.csv', header=True, inferSchema=True)
df.count()

1000000

In [5]:
df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- image_name: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [26]:
from pyspark.sql.functions import *
print(df.filter(col('amount')>0).count())
df.withColumn('transformation_amount',when(col('amount')<0,0).otherwise(col('amount'))).show()
# df.select('amount').show()

980371
+--------------+-----------+-----------+------+--------------------+---------------------+
|transaction_id|customer_id| image_name|amount|           timestamp|transformation_amount|
+--------------+-----------+-----------+------+--------------------+---------------------+
|         TXN_1|  CUST_3673|img_005.jpg|400.16|2026-09-15 07:45:...|               400.16|
|         TXN_2|  CUST_1159|img_003.jpg|232.09|2026-08-19 09:23:...|               232.09|
|         TXN_3|  CUST_4530|img_003.jpg| 59.02|2026-08-18 14:05:...|                59.02|
|         TXN_4|  CUST_1516|img_001.jpg| 40.36|2026-08-28 10:35:...|                40.36|
|         TXN_5|  CUST_4792|img_005.jpg|108.95|2026-09-11 20:19:...|               108.95|
|         TXN_6|  CUST_4384|img_002.jpg| 19.19|2026-09-05 03:07:...|                19.19|
|         TXN_7|  CUST_2528|img_004.jpg|222.73|2026-08-23 19:32:...|               222.73|
|         TXN_8|  CUST_3514|img_002.jpg|179.79|2026-09-14 03:54:...|               

In [15]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf
from pyspark.sql.types import DoubleType

# Initialize Spark with Arrow configuration enabled
spark = SparkSession.builder \
    .appName("AdvancedPySpark") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# Sample transaction data
data = [(1, 100.0), (2, 250.5), (3, 50.0)]
df = spark.createDataFrame(data, ["user_id", "amount"])

# Define a vectorized Pandas UDF to compute a custom tax formula
@pandas_udf(DoubleType())
def calculate_tax_vectorized(amounts: pd.Series) -> pd.Series:
    # This runs native, fast Pandas vector calculations under the hood
    return amounts * 0.15

# Apply the Pandas UDF
df_with_tax = df.withColumn("tax", calculate_tax_vectorized(col("amount")))
df_with_tax.show()



c:\Users\uday.bachina\airflow_folder\airflow_env\Lib\site-packages\pyspark\sql\pandas\typehints.py:60: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Py4JJavaError: An error occurred while calling o230.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 5 in stage 17.0 failed 1 times, most recent failure: Lost task 5.0 in stage 17.0 (TID 87) (FeuMT-059L-R.FEUJI.INC executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:704)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:686)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1068)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1045)
	a

In [31]:
import pandas as pd
pd.Series(df_with_tax.values).to_dict()

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `values` is not supported.

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 61823)
Traceback (most recent call last):
  File "C:\Users\uday.bachina\AppData\Local\Python\pythoncore-3.14-64\Lib\socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\uday.bachina\AppData\Local\Python\pythoncore-3.14-64\Lib\socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\uday.bachina\AppData\Local\Python\pythoncore-3.14-64\Lib\socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\uday.bachina\AppData\Local\Python\pythoncore-3.14-64\Lib\socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "c:\U

In [26]:
df_with_tax.columns

['user_id', 'amount', 'tax']